# Module 01: Exploratory Data Analysis
## Notebook 1 — Vectorized Data Wrangling, Imputation, and Memory Profiling

This notebook builds a production-style pandas pipeline on top of two datasets:

- `data/sf_salaries.csv` — San Francisco public employee compensation records
- `data/ecommerce_purchases.csv` — synthetic e-commerce transaction log

We move away from repeated, order-dependent notebook cells (`df.drop(..., inplace=True)`) toward **method-chained, vectorized pipelines** built with `.pipe()` and `.assign()`, add **conditional imputation** for missing values, and profile **memory footprint** before/after dtype optimization.

> **v2 note:** this revision folds in a technical code review -- see inline comments in `optimize_memory` and `transform_compensation_data` for the specific fixes (nullable-integer NaN handling, a categorical cardinality ceiling, and NaN-safe status filtering).

In [11]:
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# OS-agnostic relative paths. `Path.cwd()` is used rather than `__file__`
# since notebooks don't define `__file__`; run this notebook from the
# `01_exploratory_data_analysis/` directory (its own folder) so `DATA_DIR`
# resolves correctly.
DATA_DIR = Path.cwd() / "data"
SF_SALARIES_PATH = DATA_DIR / "sf_salaries.csv"
TRANSFORMED_OUT_PATH = DATA_DIR / "sf_salaries_transformed.csv"


## 1. Data Ingestion & Memory Profiling

In [12]:
def _nullable_int_dtype(min_val: float, max_val: float) -> str:
    """Smallest pandas nullable integer dtype that can hold [min_val, max_val]."""
    bounds = [
        ("Int8", np.iinfo(np.int8)),
        ("Int16", np.iinfo(np.int16)),
        ("Int32", np.iinfo(np.int32)),
        ("Int64", np.iinfo(np.int64)),
    ]
    for name, info in bounds:
        if min_val >= info.min and max_val <= info.max:
            return name
    return "Int64"


def optimize_memory(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns and convert repetitive strings to categories
    to reduce memory footprint.

    v2 fixes (code review):
      - NaN-containing integer-like float columns no longer get silently
        force-cast to plain float64 (or blow up with a ValueError on a hard
        int cast). They are cast to pandas\' *nullable* integer dtypes
        (Int8/Int16/Int32/Int64), which represent missing values as `pd.NA`
        while still saving memory relative to float64.
      - Categorical conversion enforces an absolute cardinality ceiling
        (`nunique < 500`) in addition to the relative ratio check
        (`nunique / len(df) < 0.2`), so high-cardinality ID-like columns
        (e.g. emails, free-text names) don\'t get converted into a category
        with nearly as many codes as rows, which wastes memory instead of
        saving it.
      - Checks `pd.api.types.is_string_dtype(...)` alongside `== object`,
        since pandas 2.x/3.x can back text columns with either the legacy
        `object` dtype or the newer `StringDtype` -- checking only `object`
        silently skips every string column under the newer default.
    """
    initial_mem = df.memory_usage(deep=True).sum() / 1024 ** 2

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_bool_dtype(col_type):
            continue

        if pd.api.types.is_integer_dtype(col_type):
            # Genuine numpy int dtype: by definition has no NaNs, safe to
            # downcast directly.
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            non_null = df[col].dropna()
            is_whole_number_col = not non_null.empty and (non_null % 1 == 0).all()

            if is_whole_number_col and df[col].isna().any():
                dtype = _nullable_int_dtype(non_null.min(), non_null.max())
                df[col] = df[col].astype(dtype)
            elif is_whole_number_col:
                df[col] = pd.to_numeric(df[col], downcast="integer")
            else:
                df[col] = pd.to_numeric(df[col], downcast="float")

        elif col_type == object or pd.api.types.is_string_dtype(col_type):
            num_unique = df[col].nunique(dropna=True)
            num_total = len(df[col])
            if num_total > 0 and num_unique < 500 and (num_unique / num_total) < 0.2:
                df[col] = df[col].astype("category")

    final_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    reduction = 100 * (initial_mem - final_mem) / initial_mem if initial_mem else 0
    print(
        f"Memory reduced from {initial_mem:.2f} MB to {final_mem:.2f} MB "
        f"({reduction:.1f}% reduction)"
    )
    return df


raw_salaries = pd.read_csv(SF_SALARIES_PATH)
print(raw_salaries.dtypes)
raw_salaries.head()

Id                int64
EmployeeName     object
JobTitle         object
BasePay         float64
OvertimePay     float64
OtherPay        float64
Benefits        float64
Year              int64
Status           object
dtype: object


,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,Year,Status
0,1,Employee_1,SPECIAL NURSE,91127.71,0.0,2479.79,27054.60,2022,PT
1,2,Employee_2,ANIMAL CARE ATTENDANT,54895.61,5505.3,4678.69,21211.19,2022,PT
2,3,Employee_3,HEAD LIBRARIAN,22814.01,0.0,3817.43,6813.61,2023,FT
3,4,Employee_4,SENIOR CLERK,45058.99,2729.4,1394.15,19027.54,2023,FT
4,5,Employee_5,SENIOR CLERK,127962.30,0.0,5097.18,46744.96,2022,FT


In [13]:
profiled = optimize_memory(raw_salaries.copy())
profiled.dtypes

Memory reduced from 0.65 MB to 0.28 MB (57.7% reduction)


Id                 int16
EmployeeName      object
JobTitle        category
BasePay          float64
OvertimePay      float64
OtherPay         float32
Benefits         float64
Year               int16
Status          category
dtype: object

## 2. Missingness Diagnostics & Conditional Imputation

Rather than blindly filling every numeric column with its global mean (which distorts variance and hides structure), we impute **conditionally on a relevant group** — here, `JobTitle` — falling back to the global median only when a group has no observed values at all.

In [14]:
missing_report = (
    raw_salaries.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda d: (d["missing_count"] / len(raw_salaries) * 100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)
missing_report

,missing_count,missing_pct
Benefits,248,8.27
OvertimePay,91,3.03


In [15]:
def conditional_group_impute(
    df: pd.DataFrame, target_col: str, group_col: str
) -> pd.DataFrame:
    """Impute `target_col` with the median of its `group_col` cohort;
    fall back to the global median for groups that are entirely missing."""
    df = df.copy()
    group_median = df.groupby(group_col)[target_col].transform("median")
    global_median = df[target_col].median()
    df[target_col] = df[target_col].fillna(group_median).fillna(global_median)
    return df


salaries_imputed = raw_salaries.copy()
for col in ["OvertimePay", "Benefits"]:
    salaries_imputed = conditional_group_impute(salaries_imputed, col, "JobTitle")

print("Remaining NaNs after conditional imputation:")
print(salaries_imputed[["OvertimePay", "Benefits"]].isna().sum())

Remaining NaNs after conditional imputation:
OvertimePay    0
Benefits       0
dtype: int64


## 3. Vectorized Wrangling & Method Chaining

In [16]:
def transform_compensation_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    """End-to-end transformation pipeline using pandas method chaining.

    v2 fix: the filter no longer silently drops active employees whose
    `status` is missing. `status != "TERMINATED"` evaluates to NaN (falsy in
    a boolean mask) when `status` is NaN, so a plain conjunction used to
    exclude those rows even though they were never actually confirmed
    terminated. We explicitly keep rows with an unknown status, and use
    `engine="python"` so `.query()` can evaluate the `status.isna()` method
    call (the default numexpr engine doesn\'t support method calls).
    """
    return (
        raw_df.pipe(optimize_memory)
        .rename(columns=lambda c: c.strip().lower().replace(" ", "_"))
        .assign(
            basepay=lambda d: d["basepay"].fillna(0),
            overtimepay=lambda d: d["overtimepay"].fillna(0),
            otherpay=lambda d: d["otherpay"].fillna(0),
        )
        .assign(
            total_compensation=lambda d: d["basepay"] + d["overtimepay"] + d["otherpay"],
            is_executive=lambda d: d["jobtitle"].astype(str).str.contains(
                "DIRECTOR|CHIEF|HEAD", case=False, na=False
            ),
            job_level=lambda d: np.select(
                [
                    d["total_compensation"] >= 150000,
                    d["total_compensation"] >= 75000,
                ],
                ["High", "Medium"],
                default="Standard",
            ),
        )
        .query(
            'total_compensation > 0 and (status != "TERMINATED" or status.isna())',
            engine="python",
        )
    )


salaries_transformed = transform_compensation_data(raw_salaries.copy())
salaries_transformed[["jobtitle", "total_compensation", "is_executive", "job_level"]].head(10)

Memory reduced from 0.65 MB to 0.28 MB (57.7% reduction)


,jobtitle,total_compensation,is_executive,job_level
0,SPECIAL NURSE,93607.500039,False,Medium
1,ANIMAL CARE ATTENDANT,65079.599941,False,Standard
2,HEAD LIBRARIAN,26631.439932,True,Standard
3,SENIOR CLERK,49182.540024,False,Standard
4,SENIOR CLERK,133059.480176,False,Medium
5,SOCIAL WORKER,63097.009985,False,Standard
6,SPECIAL NURSE,57726.979971,False,Standard
7,ASSISTANT MEDICAL EXAMINER,103919.760049,False,Medium
8,FIREFIGHTER,103589.870020,False,Medium
9,SPECIAL NURSE,79338.790029,False,Medium


## 4. Statistical Aggregation Engine

In [17]:
def compute_cohort_metrics(
    df: pd.DataFrame, groupby_col: str, metric_col: str
) -> pd.DataFrame:
    """Computes robust statistical metrics (median, IQR, mean, std) across segments."""
    return (
        df.groupby(groupby_col, observed=True)[metric_col]
        .agg(
            sample_count="count",
            mean="mean",
            median="median",
            std="std",
            iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
        )
        .sort_values(by="median", ascending=False)
        .reset_index()
    )


cohort_by_level = compute_cohort_metrics(salaries_transformed, "job_level", "total_compensation")
cohort_by_level

,job_level,sample_count,mean,median,std,iqr
0,High,168,171812.668748,167233.604995,18820.370495,20730.167422
1,Medium,1608,100209.049145,95524.660015,18957.791010,27255.790045
2,Standard,910,59092.526870,61486.480010,11716.646044,15947.477481


In [18]:
cohort_by_exec = compute_cohort_metrics(salaries_transformed, "is_executive", "total_compensation")
cohort_by_exec

,is_executive,sample_count,mean,median,std,iqr
0,True,633,127836.297460,126944.050020,35210.378493,47430.070039
1,False,2053,79325.135185,79211.829971,22066.549766,29551.199978


## 5. IQR Outlier Filtering

Detection thresholds are bound by $Q_1 - 1.5 \times \text{IQR}$ and $Q_3 + 1.5 \times \text{IQR}$, computed on `total_compensation` before any downstream modeling.

In [19]:
def flag_iqr_outliers(df: pd.DataFrame, col: str) -> pd.DataFrame:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return df.assign(is_outlier=lambda d: ~d[col].between(lower, upper))


salaries_flagged = flag_iqr_outliers(salaries_transformed, "total_compensation")
outlier_count = salaries_flagged["is_outlier"].sum()
print(f"Outliers flagged: {outlier_count} / {len(salaries_flagged)}")

Outliers flagged: 94 / 2686


In [20]:
# Persist the cleaned frame for the visualization notebook (pathlib, no
# hardcoded working-directory string)
salaries_transformed.to_csv(TRANSFORMED_OUT_PATH, index=False)
print(f"Saved {TRANSFORMED_OUT_PATH}:", salaries_transformed.shape)

Saved C:\Users\AliFa\Desktop\To_GitHub\Module1_Exploratory_Data_Dnalysis_1\data\sf_salaries_transformed.csv: (2686, 12)
